In [516]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GAE
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from torch_geometric.utils import from_scipy_sparse_matrix
from torch_geometric.data import Data
import scipy.sparse as sp
import networkx as nx
import matplotlib.pyplot as plt
from torch.nn import BatchNorm1d
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch.nn import BatchNorm1d
from torch_geometric.utils import dropout_adj
import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch.nn import BatchNorm1d

class GCNEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=3, dropout_rate=0.1):
        super(GCNEncoder, self).__init__()
        self.num_layers = num_layers
        self.dropout_rate = dropout_rate
        
        
        # First layer: input to 2x hidden size
        self.convs = torch.nn.ModuleList()
        self.convs.append(GCNConv(in_channels, 2 * hidden_channels, cached=False))
        
        # Add intermediate layers
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(2 * hidden_channels, 2 * hidden_channels, cached=False))
        
        # Final layer: hidden size to output size
        self.convs.append(GCNConv(2 * hidden_channels, out_channels, cached=False))
    
        # Batch normalization layers for each convolution layer
        self.bns = torch.nn.ModuleList([BatchNorm1d(2 * hidden_channels) for _ in range(num_layers - 1)])

        # Dropout rate
        self.dropout = torch.nn.Dropout(p=dropout_rate)
    
        # Activation functions
        self.activation_hidden = torch.nn.ReLU()  # For hidden layers
        self.activation_output = torch.nn.Sigmoid()  # For final layer

    def forward(self, x, edge_index):
        # Apply edge dropout (optional)
        #edge_index, _ = dropout_adj(edge_index, p=self.edge_dropout_rate, training=self.training)
        
        # First layer (without batch normalization)
        x = self.convs[0](x, edge_index)
        x = F.relu(x)
    
        # Intermediate layers with batch normalization and dropout
        for i in range(1, self.num_layers - 1):
            x_residual = x # For residual connection
            x = self.convs[i](x, edge_index)
            x = self.bns[i-1](x)  # Apply batch normalization
            x = F.relu(x)
            x = self.dropout(x)
            x = x + x_residual  # Add residual connection
    
        # Final layer
        x = self.convs[-1](x, edge_index)
        x = F.relu(x)
        
        return x

def apply_gae_to_adjacency(adj_matrix, node_features, embedding_dim=8, epochs=200, dropout_rate=0.01, weight_decay=1e-4):
    # Prepare data and adjacency matrix
    adj_sparse = sp.csr_matrix(adj_matrix)
    adj_normalized = normalize_adj(adj_sparse + sp.eye(adj_sparse.shape[0]))
    edge_index, edge_weight = from_scipy_sparse_matrix(adj_normalized)

    # Create data object
    x = torch.tensor(node_features, dtype=torch.float)
    data = Data(x=x, edge_index=edge_index)

    # Define GCN Encoder and GAE
    encoder = GCNEncoder(in_channels=x.shape[1], hidden_channels=embedding_dim, out_channels=embedding_dim, dropout_rate=dropout_rate)
    model = GAE(encoder)

    # Training settings
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    data = data.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=weight_decay)

    # Training loop
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        z = model.encode(data.x, data.edge_index)
        loss = model.recon_loss(z, data.edge_index)
        loss.backward()
        optimizer.step()

    # After training, reconstruct adjacency matrix
    model.eval()
    with torch.no_grad():
        z = model.encode(data.x, data.edge_index)
        adj_reconstructed = model.decoder.forward_all(z).cpu().numpy()
        
    return adj_reconstructed

#normalized_laplacian
def normalize_adj(adj):
    """Compute the normalized Laplacian."""
    adj = sp.coo_matrix(adj)
    rowsum = np.array(adj.sum(1)).flatten()
    d_inv_sqrt = np.power(rowsum + 1e-5, -0.5)
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.0
    d_mat_inv_sqrt = sp.diags(d_inv_sqrt)
    identity = sp.eye(adj.shape[0])  # Identity matrix of the same size as adj
    laplacian = identity - d_mat_inv_sqrt.dot(adj).dot(d_mat_inv_sqrt)  # Normalized Laplacian
    return laplacian.tocoo()


# Convert ground truth matrix to edge list
def convert_matrix_to_edge_list(matrix):
    edges = []
    for source in matrix.index:
        for target in matrix.columns:
            if matrix.at[source, target] == 1:
                edges.append((source, target))
    return edges

# Function to extract gene interactions (edges) from a binarized matrix
def extract_gene_interactions(final_matrix, gene_names):
    edges = []
    num_genes = final_matrix.shape[0]

    # Loop through the upper triangle of the matrix to avoid duplicates (since it's undirected)
    for i in range(num_genes):
        for j in range(i + 1, num_genes):
            if final_matrix[i, j] == 1:  # Check if there's an interaction
                edges.append(f"{gene_names[i]}-{gene_names[j]}")
    
    return edges


# Function to convert edges to consistent tuple format (e.g., ('G1', 'G10'))
def convert_edges_to_tuples(edge_list):
    edge_tuples = []
    for edge in edge_list:
        gene1, gene2 = edge.split('-')
        # Sort genes to maintain consistency in the edge representation (G1, G10) or (G10, G1)
        edge_tuples.append(tuple(sorted([gene1, gene2])))
    return set(edge_tuples)

# Function to compare predicted edges with known ground truth and generate the metrics
def compare_with_ground_truth(predicted_edges, known_edges_set):
    predicted_set = convert_edges_to_tuples(predicted_edges)
    
    # Calculate true positives and false positives
    true_positives = predicted_set.intersection(known_edges_set)
    false_positives = predicted_set.difference(known_edges_set)
    
    # Metrics
    num_true_positives = len(true_positives)
    num_false_positives = len(false_positives)
    
    return true_positives, false_positives, num_true_positives, num_false_positives

In [549]:
# Load your data (100x1700 matrix)
data = pd.read_csv("/Users/nirwantandukar/input_data/Database.DREAM4/NORMALIZED_EXP/normalized_weight_matrix_all_GRN_100_01_0.2filter.csv", index_col=0)

# Get the unique algorithm names
algorithms = sorted(set([col.split('.')[0] for col in data.columns]))

# Sum adjacency matrices
def sum_adjacency_matrices(data, algorithms):
    sum_matrix = np.zeros((100, 100))  # Initialize sum matrix
    for algorithm in algorithms:
        if algorithm == 'GRN_MRNET':
            alg_columns = [col for col in data.columns if col.startswith('GRN_MRNET.') and not col.startswith('GRN_MRNETB')]
        else:
            alg_columns = [col for col in data.columns if col.startswith(algorithm)]
        
        alg_matrix = data[alg_columns].values
        sum_matrix += alg_matrix
        
    return sum_matrix

summed_matrix = sum_adjacency_matrices(data, algorithms)

# Binarize summed matrix
def binarize_summed_matrix(summed_matrix, thresholds):
    binarized_matrices = {}
    for threshold in thresholds:
        binarized_matrix = (summed_matrix >= threshold).astype(int)
        binarized_matrices[f'summed_matrix_{threshold}'] = binarized_matrix
    return binarized_matrices

thresholds = [2, 4, 6, 8, 10, 12, 14]
binarized_matrices = binarize_summed_matrix(summed_matrix, thresholds)

# Run GCN multiple times
def run_gcn_multiple_times(binarized_matrices, node_features, embedding_dim=8, epochs=200, dropout_rate=0.5, num_iterations=10):
    gcn_results = {}
    for threshold, binarized_matrix in binarized_matrices.items():
        accumulated_matrix = np.zeros_like(binarized_matrix, dtype=float)
        for iteration in range(num_iterations):
            gcn_weight_matrix = apply_gae_to_adjacency(binarized_matrix, node_features, embedding_dim, epochs=epochs, dropout_rate=dropout_rate)
            accumulated_matrix += gcn_weight_matrix
        averaged_matrix = accumulated_matrix / num_iterations
        gcn_results[f'GCN_weight_matrix_{threshold.split("_")[-1]}'] = averaged_matrix
    return gcn_results

num_genes = binarized_matrices['summed_matrix_2'].shape[0]
node_features = np.identity(num_genes)

gcn_results = run_gcn_multiple_times(binarized_matrices, node_features, embedding_dim=8, epochs=200, dropout_rate=0.01, num_iterations=10)



In [559]:
import numpy as np

# Step 1: Binarize each GCN matrix individually

def binarize_each_gcn_matrix(gcn_results, threshold=0.6):
    binarized_matrices = {}
    for matrix_name, gcn_matrix in gcn_results.items():
        # Apply the binarization to each matrix separately
        binarized_matrix = (gcn_matrix >= threshold).astype(int)
        binarized_matrices[matrix_name] = binarized_matrix
        
        # Print the results for each matrix
        print(f"Binarized matrix for {matrix_name} with threshold {threshold}:\n{binarized_matrix}")
        print(f"Shape: {binarized_matrix.shape}")
    
    return binarized_matrices

# Apply the binarization to each GCN weight matrix
binarization_threshold = 0.6
binarized_gcn_matrices = binarize_each_gcn_matrix(gcn_results, threshold=binarization_threshold)

# Now each matrix (e.g., GCN_weight_matrix_2, GCN_weight_matrix_4) has been binarized individually


Binarized matrix for GCN_weight_matrix_2 with threshold 0.6:
[[1 0 0 ... 0 1 1]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [1 0 0 ... 0 1 1]
 [1 0 0 ... 0 1 1]]
Shape: (100, 100)
Binarized matrix for GCN_weight_matrix_4 with threshold 0.6:
[[1 0 0 ... 0 0 0]
 [0 1 0 ... 0 0 0]
 [0 0 1 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 1 1]
 [0 0 0 ... 0 1 1]]
Shape: (100, 100)
Binarized matrix for GCN_weight_matrix_6 with threshold 0.6:
[[1 0 0 ... 0 0 0]
 [0 1 0 ... 0 0 0]
 [0 0 1 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 1 1]
 [0 0 0 ... 0 1 1]]
Shape: (100, 100)
Binarized matrix for GCN_weight_matrix_8 with threshold 0.6:
[[1 0 0 ... 0 0 0]
 [0 1 0 ... 0 0 0]
 [0 0 1 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 1 1]
 [0 0 0 ... 0 1 1]]
Shape: (100, 100)
Binarized matrix for GCN_weight_matrix_10 with threshold 0.6:
[[1 0 0 ... 0 0 0]
 [0 1 0 ... 0 0 0]
 [0 0 1 ... 0 0 1]
 ...
 [0 0 0 ... 1 0 0]
 [0 0 0 ... 0 1 0]
 [0 0 1 ... 0 0 1]]
Shape: (100, 100)
Bina

In [561]:
# Function to extract gene interactions (edges) from a binarized matrix
def extract_gene_interactions(binarized_matrix, gene_names):
    interactions = []
    num_genes = len(gene_names)
    
    for i in range(num_genes):
        for j in range(i + 1, num_genes):  # Upper triangular part to avoid duplicates
            if binarized_matrix[i, j] == 1:
                interactions.append((gene_names[i], gene_names[j]))  # Gene-gene interaction
    
    return interactions

# Function to apply to each binarized matrix and extract gene interactions
def extract_interactions_from_binarized_matrices(binarized_matrices, gene_names):
    gene_interactions = {}
    
    for matrix_name, binarized_matrix in binarized_matrices.items():
        # Extract the gene-gene interactions from each binarized matrix
        interactions = extract_gene_interactions(binarized_matrix, gene_names)
        gene_interactions[matrix_name] = interactions
        
        # Print or save the gene interactions for each matrix
        print(f"Gene interactions for {matrix_name}:")
        print(interactions)
    
    return gene_interactions

# Apply the extraction to each binarized matrix
gene_interactions = extract_interactions_from_binarized_matrices(binarized_gcn_matrices, gene_names)

# Optionally, save the interactions as CSV files (if needed)
# for matrix_name, interactions in gene_interactions.items():
#     pd.DataFrame(interactions, columns=['Gene1', 'Gene2']).to_csv(f'gene_interactions_{matrix_name}.csv', index=False)


Gene interactions for GCN_weight_matrix_2:
[('G1', 'G11'), ('G1', 'G12'), ('G1', 'G15'), ('G1', 'G16'), ('G1', 'G17'), ('G1', 'G18'), ('G1', 'G19'), ('G1', 'G2'), ('G1', 'G20'), ('G1', 'G21'), ('G1', 'G22'), ('G1', 'G23'), ('G1', 'G24'), ('G1', 'G25'), ('G1', 'G26'), ('G1', 'G28'), ('G1', 'G29'), ('G1', 'G3'), ('G1', 'G31'), ('G1', 'G33'), ('G1', 'G36'), ('G1', 'G37'), ('G1', 'G4'), ('G1', 'G41'), ('G1', 'G42'), ('G1', 'G43'), ('G1', 'G44'), ('G1', 'G45'), ('G1', 'G46'), ('G1', 'G47'), ('G1', 'G48'), ('G1', 'G49'), ('G1', 'G5'), ('G1', 'G50'), ('G1', 'G51'), ('G1', 'G52'), ('G1', 'G53'), ('G1', 'G54'), ('G1', 'G56'), ('G1', 'G57'), ('G1', 'G58'), ('G1', 'G59'), ('G1', 'G6'), ('G1', 'G60'), ('G1', 'G61'), ('G1', 'G62'), ('G1', 'G63'), ('G1', 'G64'), ('G1', 'G65'), ('G1', 'G67'), ('G1', 'G68'), ('G1', 'G69'), ('G1', 'G70'), ('G1', 'G71'), ('G1', 'G72'), ('G1', 'G73'), ('G1', 'G74'), ('G1', 'G75'), ('G1', 'G76'), ('G1', 'G77'), ('G1', 'G79'), ('G1', 'G8'), ('G1', 'G80'), ('G1', 'G81'), ('

In [563]:
# Load ground truth data
file_path = "/Users/nirwantandukar/input_data/Database.DREAM4/GS/dream4_100_01_gs.csv"
ground_truth = pd.read_csv(file_path, index_col=0)
print(f"Ground truth loaded with shape: {ground_truth.shape}")

# Convert ground truth to edges
known_edges_set = set(convert_matrix_to_edge_list(ground_truth))
print(f"Number of known edges: {len(known_edges_set)}")


Ground truth loaded with shape: (100, 100)
Number of known edges: 175


In [565]:
# Function to convert predicted edges to consistent tuple format (e.g., ('GeneA', 'GeneB'))
def convert_edges_to_tuples(edge_list):
    edge_tuples = []
    for gene1, gene2 in edge_list:
        edge_tuples.append(tuple(sorted([gene1, gene2])))  # Sort for consistency
    return set(edge_tuples)

# Function to compare predicted edges with known ground truth and generate metrics
def compare_with_ground_truth(predicted_edges, known_edges_set):
    # Convert predicted edges to tuples
    predicted_set = convert_edges_to_tuples(predicted_edges)
    
    # Calculate true positives (TPs) and false positives (FPs)
    true_positives = predicted_set.intersection(known_edges_set)
    false_positives = predicted_set.difference(known_edges_set)
    
    # Metrics
    num_true_positives = len(true_positives)
    num_false_positives = len(false_positives)
    
    return num_true_positives, num_false_positives, true_positives, false_positives

# Compare gene interactions with the ground truth for each threshold
comparison_results = {}

for matrix_name, predicted_edges in gene_interactions.items():
    print(f"Comparing predicted edges for {matrix_name}...")
    
    # Compare predicted edges with ground truth
    num_true_positives, num_false_positives, true_positives, false_positives = compare_with_ground_truth(predicted_edges, known_edges_set)
    
    # Store comparison results
    comparison_results[matrix_name] = {
        'True_Positives': num_true_positives,
        'False_Positives': num_false_positives,
        'True_Positive_Interactions': list(true_positives),
        'False_Positive_Interactions': list(false_positives)
    }

    # Print summary
    print(f"{matrix_name}: TP = {num_true_positives}, FP = {num_false_positives}")

# Optionally, you can save the results to a CSV or JSON file for later analysis.


Comparing predicted edges for GCN_weight_matrix_2...
GCN_weight_matrix_2: TP = 62, FP = 1224
Comparing predicted edges for GCN_weight_matrix_4...
GCN_weight_matrix_4: TP = 54, FP = 845
Comparing predicted edges for GCN_weight_matrix_6...
GCN_weight_matrix_6: TP = 51, FP = 752
Comparing predicted edges for GCN_weight_matrix_8...
GCN_weight_matrix_8: TP = 45, FP = 551
Comparing predicted edges for GCN_weight_matrix_10...
GCN_weight_matrix_10: TP = 28, FP = 399
Comparing predicted edges for GCN_weight_matrix_12...
GCN_weight_matrix_12: TP = 13, FP = 218
Comparing predicted edges for GCN_weight_matrix_14...
GCN_weight_matrix_14: TP = 10, FP = 254
